# FISH bulk processing
From microscope images to mRNA counts per cell (and per mother and bud):

1. **Read** the .vsi and DIC images (several images at the same time)
2. **Segment** cells and nuclei (Omnipose, on the GPU if available)
3. **Detect spots**, align the masks, assign buds to mothers (bud model) and count spots per cell
4. **Check and correct** in napari
5. **Reassign spots** to the corrected masks

Each step writes its results to the image's folder in `output/` and records what it did in `img.json`, so you can
stop after any step and continue later.

The work for one image is in short functions in `buddy/pipeline.py` (one per step). The napari
**buddy pipeline** panel (Plugins → napari-buddy) runs the same functions, so both give the same result.

In [1]:
# All paths in this notebook start from ROOT, the folder that holds the buddy and napari-buddy repositories and
# the example data ("test yourself"); see TUTORIAL.md, section 1.1. Change ROOT if your layout is different.
from pathlib import Path
ROOT = Path.cwd().resolve().parents[1]
assert (ROOT / "buddy").is_dir(), f"ROOT = {ROOT} does not contain the buddy repository: set ROOT by hand"
print("ROOT =", ROOT)


ROOT = F:\buddy-code-for-submission


In [2]:
import warnings
SHOW_WARNINGS = True       # False silences all Python warnings in this notebook (they are only hidden, not fixed)
if not SHOW_WARNINGS:
    warnings.filterwarnings("ignore")


In [3]:
import time, logging
from pathlib import Path
from buddy.experiment import Experiment
from buddy.image import Image
from buddy import pipeline                    # the processing steps for one image
from buddy.which_images import select_images   # skips images already processed with the same settings
from buddy.parallel import default_workers, run_parallel, read_one, spots_one
WORKERS = default_workers(gb_per_image=4)     # images read at the same time (limited by CPU cores and free memory)
print('parallel workers:', WORKERS)

# Spot detection (step 3 below): None = automatic (one image at a time on the GPU when GPU spot detection is on,
# otherwise several images in parallel); True / False = always parallel / always one at a time
PARALLEL_SPOTS = None

parallel workers: 3


***

### 1. Read in microscope images

Each step skips images that were already processed with the same settings. Images processed with other
settings are listed with what changed, and you are asked whether to overwrite them (config: `"if_already_processed": "ask" | "skip" | "overwrite"`).


In [4]:
# Your data: a folder with the microscope files (.vsi + _DIC.tif, or TIFF stacks) and the config file for that
# experiment. The results go to DATA / "output". For a new experiment, point these two lines at your folders, e.g.
#   DATA = Path(r"E:\smFISH\251101 YET915 BR6")   and   CFG_FILE = DATA / "config.json"
DATA = ROOT / "test yourself/small example set/YET572 BR4"
CFG_FILE = ROOT / "buddy/config_files/config_example.json"   # thresholds, Omnipose settings and all other settings

exp = Experiment.from_cfg_file(CFG_FILE)
exp.inputdir = DATA
exp.outputdir = DATA / "output"
exp.create_image_list()


10/29/2026 11:41:56AM [INFO] found cfg file F:\buddy-code-for-submission\buddy\config_files\config_example.json
10/29/2026 11:41:56AM [INFO] getting .vsi files from F:\buddy-code-for-submission\test yourself\small example set\YET572 BR4
10/29/2026 11:41:56AM [INFO] examining image F:\buddy-code-for-submission\test yourself\small example set\YET572 BR4\YET572_BR1_ASH1CLB2Q670_SRL1cal610_SUN4Q570_03_CY5, CY3.5 NAR, CY3, DAPI.vsi
10/29/2026 11:41:56AM [INFO] ..found VSI directory _YET572_BR1_ASH1CLB2Q670_SRL1cal610_SUN4Q570_03_CY5, CY3.5 NAR, CY3, DAPI_
10/29/2026 11:41:56AM [INFO] ..found DIC file YET572_BR1_ASH1CLB2Q670_SRL1cal610_SUN4Q570_03_DIC.tif
10/29/2026 11:41:57AM [INFO] ..all good, adding image to image list
10/29/2026 11:41:57AM [INFO] examining image F:\buddy-code-for-submission\test yourself\small example set\YET572 BR4\YET572_BR1_ASH1CLB2Q670_SRL1cal610_SUN4Q570_02_CY5, CY3.5 NAR, CY3, DAPI.vsi
10/29/2026 11:41:57AM [INFO] ..found VSI directory _YET572_BR1_ASH1CLB2Q670_SRL1

In [5]:
mode = f'parallel ({WORKERS} workers)'
to_process = select_images(exp, 'read')       # new images + (if you agree) images processed with other settings
jobs = [(params, CFG_FILE, exp.inputdir, exp.outputdir, mode) for params in to_process]
run_parallel(read_one, jobs, WORKERS)

step 1 (read): 2 new, 0 already processed with these settings (skipped), 0 processed with other settings
  processing 2 image(s)
  1/2 done: YET572_BR1_ASH1CLB2Q670_SRL1cal610_SUN4Q570_02
  2/2 done: YET572_BR1_ASH1CLB2Q670_SRL1cal610_SUN4Q570_03


['F:\\buddy-code-for-submission\\test yourself\\small example set\\YET572 BR4\\output_052\\YET572_BR1_ASH1CLB2Q670_SRL1cal610_SUN4Q570_02\\img.json',
 'F:\\buddy-code-for-submission\\test yourself\\small example set\\YET572 BR4\\output_052\\YET572_BR1_ASH1CLB2Q670_SRL1cal610_SUN4Q570_03\\img.json']

***

### 2. Segment images (GPU, one image after the other)
Per image (`pipeline.segment_image`): cell masks from a DIC + DAPI composite and nucleus masks from DAPI
(Omnipose), then small objects are removed and, if set, cells touching the image edge (`DIC_masks_pp.tif`).

In [6]:
exp.read_image_list_from_jsons()
to_process = select_images(exp, 'segment')
if to_process:
    exp.init_omnipose()        # runs on the GPU when "use_gpu": true in the config and a GPU is available

for n, f in enumerate(to_process, start=1):
    pipeline.segment_image(f, exp)     # segment_cells -> segment_dapi -> postprocess_masks
    print(f'  {n}/{len(to_process)} done: {Path(f).parent.name}', flush=True)

10/29/2026 11:42:49AM [INFO] reading json file F:\buddy-code-for-submission\test yourself\small example set\YET572 BR4\output_052\YET572_BR1_ASH1CLB2Q670_SRL1cal610_SUN4Q570_02\img.json
10/29/2026 11:42:49AM [INFO] reading json file F:\buddy-code-for-submission\test yourself\small example set\YET572 BR4\output_052\YET572_BR1_ASH1CLB2Q670_SRL1cal610_SUN4Q570_03\img.json
step 2 (segment): 2 new, 0 already processed with these settings (skipped), 0 processed with other settings
  processing 2 image(s)


C:\Users\Sander\anaconda3\envs\buddy052\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


10/29/2026 11:43:29AM [INFO] loading Omnipose models (cells: cyto2_omni, nuclei: nuclei) on GPU
2026-10-29 11:43:29,382	[INFO]     cellpose_omni\models.py       __init__....()	 line 440	>>cyto2_omni<< model set to be used
2026-10-29 11:43:29,573	[INFO]     cellpose_omni\core.py         assi...evice()	 line 112	Using GPU.
2026-10-29 11:43:29,575	[INFO]                                   _loc...ision()	 line  89	Enforcing deterministic FP32 CUDA kernels for reproducibility.
2026-10-29 11:43:29,748	[INFO]                                   __init__....()	 line 218	u-net config: ([2, 32, 64, 128, 256], 4, 2)
2026-10-29 11:43:30,033	[INFO]     cellpose_omni\models.py                   	 line 440	>>nuclei<< model set to be used
2026-10-29 11:43:30,037	[INFO]     cellpose_omni\core.py         assi...evice()	 line 112	Using GPU.
2026-10-29 11:43:30,177	[INFO]                                   __init__....()	 line 218	u-net config: ([2, 32, 64, 128, 256], 3, 2)
10/29/2026 11:43:30AM [INFO] found 

***

### 3. Detect spots, then assign buds + count spots
**3a** (`pipeline.spots_image`), for all images: in-focus planes, background filter, spot detection,
decomposition of dense regions, and alignment of the cell masks to the spots (on the GPU one image at a time, or
in parallel without a GPU - see `PARALLEL_SPOTS`).

**3b** (`pipeline.buds_and_counts`), per image: the bud model makes the bud masks (if enabled in the config),
then the spots are counted per cell, mother and bud (`{channel}.csv`).

In [7]:
from buddy.experiment import gpu_available
exp.read_image_list_from_jsons()
to_process = select_images(exp, 'spots')
gpu_spots = exp.parameters.get('gpu_spot_detection', True) and gpu_available(exp.use_gpu)
parallel_spots = (not gpu_spots) if PARALLEL_SPOTS is None else PARALLEL_SPOTS
mode = (f'parallel ({WORKERS} workers)' if parallel_spots else 'serial') + (' + GPU spot detection' if gpu_spots else '')
print('step 3:', mode)

# 3a: find_focus -> filter -> detect_spots -> decompose_spots -> align
jobs = [(f, CFG_FILE, exp.inputdir, exp.outputdir, mode) for f in to_process]
if parallel_spots:
    run_parallel(spots_one, jobs, WORKERS)
else:
    for n, f in enumerate(to_process, start=1):
        pipeline.spots_image(f, exp, mode)
        print(f'  {n}/{len(to_process)} done: {Path(f).parent.name}', flush=True)

# 3b: classify_buds (bud model) -> assign_spots
exp.init_bud_model()       # loads the bud model once (only if bud classification is enabled in the config)
for n, f in enumerate(to_process, start=1):
    pipeline.buds_and_counts(f, exp)
    print(f'  {n}/{len(to_process)} buds + counts done: {Path(f).parent.name}', flush=True)

10/29/2026 11:45:07AM [INFO] reading json file F:\buddy-code-for-submission\test yourself\small example set\YET572 BR4\output_052\YET572_BR1_ASH1CLB2Q670_SRL1cal610_SUN4Q570_02\img.json
10/29/2026 11:45:07AM [INFO] reading json file F:\buddy-code-for-submission\test yourself\small example set\YET572 BR4\output_052\YET572_BR1_ASH1CLB2Q670_SRL1cal610_SUN4Q570_03\img.json
step 3 (spots): 2 new, 0 already processed with these settings (skipped), 0 processed with other settings
  processing 2 image(s)
step 3: serial + GPU spot detection
10/29/2026 11:45:07AM [INFO] found output dir F:\buddy-code-for-submission\test yourself\small example set\YET572 BR4\output_052\YET572_BR1_ASH1CLB2Q670_SRL1cal610_SUN4Q570_02
10/29/2026 11:45:07AM [INFO] ..DIC_corrected_masks.tif not there yet (made by step 3 (spots, alignment) or step 4 (napari))
10/29/2026 11:45:07AM [INFO] ..DAPI_corrected_masks.tif not there yet (made by step 3 (spots) or step 4 (napari))
10/29/2026 11:45:07AM [INFO] ..bud_corrected_mas

***

### 4. Check and correct
Run the cell below: napari opens the first image with a **Correct images** panel
(**◀ previous** · **save** · **save + next ▶** · **skip ▶**).

For each image, check the boxed cells (`check these buds`; hover a box for the reason) and the `bud → mother links`,
and fix what is wrong: paint in `bud masks` (1 = bud, 3 = mother), or draw a line from a bud to its mother in the
**Reassign buds to mothers** panel. Then press **save + next**.
The **Layer groups** panel shows / hides groups of layers.


In [9]:
from napari_buddy import review_images

review_images(exp, start=0)     # start = image number to begin with (0 = first)

10/29/2026 11:49:46AM [INFO] created output dir F:\buddy-code-for-submission\test yourself\small example set\YET572 BR4\output_052\YET572_BR1_ASH1CLB2Q670_SRL1cal610_SUN4Q570_03
image 1/2: YET572_BR1_ASH1CLB2Q670_SRL1cal610_SUN4Q570_03
  bud model result loaded; cells to check: 1 bud without mother; high ASH1/CLB2: 3 high ASH1/CLB2, no bud, 2 late-M pair?
  opened in 8.1 s (files 0.8 s, napari 6.6 s, other 0.8 s)


10/29/2026 12:00:13PM [INFO] found cfg file F:\buddy-code-for-submission\buddy\config_files\config_example.json
10/29/2026 12:00:13PM [INFO] getting .vsi files from F:\buddy-code-for-submission\buddy\notebooks\..\test yourself\small example set\YET572 BR4
10/29/2026 12:03:12PM [INFO] found cfg file F:\buddy-code-for-submission\buddy\config_files\config_example.json
10/29/2026 12:03:12PM [INFO] getting .vsi files from F:\buddy-code-for-submission\buddy\test yourself\small example set\YET572 BR4
10/29/2026 12:03:45PM [INFO] found cfg file F:\buddy-code-for-submission\buddy\config_files\config_example.json
10/29/2026 12:03:45PM [INFO] getting .vsi files from F:\buddy-code-for-submission\test yourself\small example set\YET572 BR4
10/29/2026 12:03:45PM [INFO] examining image F:\buddy-code-for-submission\test yourself\small example set\YET572 BR4\YET572_BR1_ASH1CLB2Q670_SRL1cal610_SUN4Q570_03_CY5, CY3.5 NAR, CY3, DAPI.vsi
10/29/2026 12:03:45PM [INFO] ..found VSI directory _YET572_BR1_ASH1CLB

***

### 5. Reassign spots
Count the spots again with the corrected masks from step 4 (`pipeline.reassign_image`).

In [ ]:
exp.read_image_list_from_jsons()
for n, f in enumerate(exp.json_files, start=1):
    pipeline.reassign_image(f, exp)      # reads corrected/ masks, rewrites {channel}.csv
    print(f'  {n}/{len(exp.json_files)} done: {Path(f).parent.name}', flush=True)